# Лабораторна робота № 3
## Перевірка статистичних гіпотез:
### гіпотеза однорідності (критерій пустих блоків),
### гіпотеза незалежності (критерії Спірмена та Кендалла),
### гіпотеза випадковості (критерій на основі кількості інверсій)

**Рівень значимості:** $\alpha = 0.05$


In [ ]:
# ── Імпорти ────────────────────────────────────────────────────────────────
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from itertools import combinations

# Фіксуємо генератор для відтворюваності результатів
np.random.seed(42)

# ── Глобальні параметри ─────────────────────────────────────────────────────
ALPHA = 0.05   # рівень значимості для всіх тестів
NS    = [50, 100, 200]   # розміри вибірок


---
## Завдання 1. Критерій пустих блоків (гіпотеза однорідності)

**Ідея:** Вибірка $X_1,\ldots,X_m$ задає $m+1$ інтервалів (блоків) на числовій прямій.  
$R$ — кількість **порожніх блоків**, у які не потрапив жоден елемент вибірки $Y_1,\ldots,Y_n$.

**Статистика під $H_0$ (однорідність):**
$$\mathbb{E}_0[R] = (m+1)\left(\frac{m}{m+1}\right)^n, \quad
  Z = \frac{R - \mathbb{E}_0[R]}{\sqrt{\mathbb{D}_0[R]}} \xrightarrow{d} N(0,1)$$

**Генерація:**  
- Вибірка X: $X_i \sim U[0,1]$ (рівномірний розподіл)  
- Вибірка Y:  
  - $H_0$ вірна: $Y_j \sim U[0,1]$  
  - $H_0$ хибна: $Y_j \sim U[0,2]$ (інший розподіл)

Перевіряємо при $m = n =$ а) 50, b) 100, с) 200.


In [ ]:
def empty_blocks_test(m: int, n: int, lam1: float = 1.0, lam2: float = 1.0,
                      same_dist: bool = True) -> dict:
    """
    Критерій пустих блоків для перевірки гіпотези однорідності.
    
    Порядок дій:
      1. Генеруємо X ~ U[0,1] (розмір m) і Y ~ U[0,1] або U[0,2] (розмір n)
      2. Сортуємо X → m+1 блоків
      3. Рахуємо кількість R пустих блоків (без жодного Y)
      4. Стандартизуємо та порівнюємо з N(0,1)
    
    same_dist=True  → вибірки з одного розподілу (H₀ вірна)
    same_dist=False → Y з U[0,2], X з U[0,1] (H₀ хибна)
    """
    # 1. Генерація вибірок
    X = np.random.uniform(0, 1, m)            # X ~ U[0,1] завжди
    if same_dist:
        Y = np.random.uniform(0, 1, n)        # Y ~ U[0,1] — H₀ вірна
    else:
        Y = np.random.uniform(0, 2, n)        # Y ~ U[0,2] — H₀ хибна
    
    # 2. Сортуємо X: x_(1) ≤ x_(2) ≤ ... ≤ x_(m)
    X_sorted = np.sort(X)
    
    # 3. Рахуємо пусті блоки
    #    Блоки: (-∞, x_(1)), (x_(1), x_(2)), ..., (x_(m), +∞)
    R = 0
    for i in range(m + 1):
        lo = X_sorted[i - 1] if i > 0 else -np.inf   # ліва межа блоку
        hi = X_sorted[i]     if i < m else  np.inf    # права межа блоку
        # Блок порожній, якщо жоден Y_j не потрапляє в (lo, hi)
        if not np.any((Y > lo) & (Y < hi)):
            R += 1
    
    # 4. Математичне сподівання та дисперсія R під H₀
    q  = m / (m + 1)                          # ймовірність що Y_j НЕ в конкретному блоці
    ER = (m + 1) * q ** n                     # E₀[R]
    DR = (                                    # D₀[R] (формула з лекції)
          (m + 1) * q**n * (1 - q**n)
        + m * (m + 1) * ((m - 1) / (m + 1))**n
        - m * (m + 1) * q**(2 * n)
    )
    DR = max(DR, 1e-10)                        # захист від нуля
    
    # 5. Стандартизована статистика та критерій
    Z        = (R - ER) / np.sqrt(DR)
    critical = stats.norm.ppf(1 - ALPHA / 2)  # ≈ 1.96 для α=0.05
    reject   = abs(Z) > critical
    
    return {"m": m, "n": n, "R": R, "ER": ER, "DR": DR,
            "Z": Z, "critical": critical, "reject": reject}

# ── Друк результатів ────────────────────────────────────────────────────────
print("=" * 90)
print("ЗАВДАННЯ 1. КРИТЕРІЙ ПУСТИХ БЛОКІВ (ОДНОРІДНІСТЬ)")
print("=" * 90)

for size in NS:
    print(f"\n{'─'*80}")
    r0 = empty_blocks_test(size, size, same_dist=True)
    r1 = empty_blocks_test(size, size, same_dist=False)
    v0 = "ВІДХИЛИТИ H₀ ✗" if r0['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1 = "ВІДХИЛИТИ H₀ ✗" if r1['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  m = n = {size}")
    print(f"  H₀ вірна  (X,Y ~ U[0,1]) | R={r0['R']:3d}  E[R]={r0['ER']:.2f}  "
          f"Z={r0['Z']:+.4f}  крит.=±{r0['critical']:.3f}  → {v0}")
    print(f"  H₀ хибна  (X~U[0,1], Y~U[0,2]) | R={r1['R']:3d}  E[R]={r1['ER']:.2f}  "
          f"Z={r1['Z']:+.4f}  крит.=±{r1['critical']:.3f}  → {v1}")


In [ ]:
# ── Візуалізація: розміщення Y-спостережень відносно X-блоків ──────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col, size in enumerate(NS):
    X_s = np.sort(np.random.uniform(0, 1, size))

    for row, (Y_arr, ttl) in enumerate([
            (np.random.uniform(0, 1, size), f"H₀ вірна (Y~U[0,1]),  m=n={size}"),
            (np.random.uniform(0, 2, size), f"H₀ хибна (Y~U[0,2]),  m=n={size}")]):
        ax = axes[row][col]
        
        # Малюємо межі блоків (порядкові статистики X)
        for xv in X_s:
            ax.axvline(xv, color='gray', lw=0.4, alpha=0.5)
        
        # Малюємо Y-спостереження
        ax.scatter(Y_arr, np.ones_like(Y_arr)*0.5, s=8,
                   color='red' if row else 'blue', alpha=0.6, label='Y')
        ax.set_xlim(-0.1, 2.1 if row else 1.1)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_title(ttl, fontsize=8)
        ax.set_xlabel('значення')
        ax.legend(fontsize=7)

plt.suptitle('Завдання 1: розміщення Y відносно блоків X', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Завдання 2. Гіпотеза незалежності: критерії Спірмена та Кендалла

**Генерація вибірки:**  
- $X_i \sim U[0,1]$  
- а) $Y_i = X_i + \varepsilon_i$, де $\varepsilon_i \sim U[0,1]$ (залежність!)  
- b) $Y_i = \varepsilon_i \sim U[0,1]$ (незалежні від X)

### А. Критерій Спірмена

$\rho_s = 1 - \dfrac{6 \sum_i (R_i - S_i)^2}{n(n^2-1)}$, де $R_i, S_i$ — ранги $X_i$ та $Y_i$.

Статистика: $T = \rho_s \sqrt{\dfrac{n-2}{1-\rho_s^2}} \sim t(n-2)$ під $H_0$.

### B. Критерій Кендалла

$\tau = \dfrac{P - Q}{n(n-1)/2}$, де $P$ — кількість узгоджених пар, $Q$ — незгоджених.

Стандартизована статистика: $Z = \dfrac{\tau}{\sqrt{2(2n+5)/(9n(n-1))}} \to N(0,1)$ під $H_0$.


In [ ]:
def spearman_test(n: int, dependent: bool = True) -> dict:
    """
    Критерій Спірмена для перевірки гіпотези незалежності.
    
    dependent=True  → Y_i = X_i + ε_i  (корельовані, H₀ хибна)
    dependent=False → Y_i = ε_i         (незалежні, H₀ вірна)
    """
    # 1. Генерація пари (X, Y)
    X   = np.random.uniform(0, 1, n)
    eps = np.random.uniform(0, 1, n)
    Y   = X + eps if dependent else eps     # залежні або незалежні
    
    # 2. Ранговий коефіцієнт кореляції Спірмена
    rho, _  = stats.spearmanr(X, Y)
    
    # 3. Тестова статистика: T = ρ√((n−2)/(1−ρ²)) ~ t(n−2) під H₀
    T = rho * np.sqrt((n - 2) / max(1 - rho**2, 1e-12))
    
    # 4. Критичне значення t-розподілу з (n−2) ступенями свободи
    critical = stats.t.ppf(1 - ALPHA / 2, df=n - 2)
    reject   = abs(T) > critical
    
    return {"n": n, "rho": rho, "T": T, "critical": critical, "reject": reject}


def kendall_test(n: int, dependent: bool = True) -> dict:
    """
    Критерій Кендалла для перевірки гіпотези незалежності.
    
    dependent=True  → Y_i = X_i + ε_i  (корельовані, H₀ хибна)
    dependent=False → Y_i = ε_i         (незалежні, H₀ вірна)
    """
    # 1. Генерація пари (X, Y)
    X   = np.random.uniform(0, 1, n)
    eps = np.random.uniform(0, 1, n)
    Y   = X + eps if dependent else eps
    
    # 2. Коефіцієнт Кендалла τ та стандартна похибка
    tau, _  = stats.kendalltau(X, Y)
    
    # 3. Стандартна похибка τ під H₀ (асимптотична формула)
    se = np.sqrt(2 * (2 * n + 5) / (9 * n * (n - 1)))
    
    # 4. Статистика Z → N(0,1) під H₀
    Z        = tau / se
    critical = stats.norm.ppf(1 - ALPHA / 2)   # ≈ 1.96
    reject   = abs(Z) > critical
    
    return {"n": n, "tau": tau, "Z": Z, "critical": critical, "reject": reject}

# ── Друк результатів ────────────────────────────────────────────────────────
print("=" * 90)
print("ЗАВДАННЯ 2. ГІПОТЕЗА НЕЗАЛЕЖНОСТІ")
print("=" * 90)

for n in NS:
    print(f"\n{'─'*80}")
    print(f"  n = {n}")
    
    # ── Спірмен ──
    r0s = spearman_test(n, dependent=False)   # H₀ вірна
    r1s = spearman_test(n, dependent=True)    # H₀ хибна
    v0s = "ВІДХИЛИТИ H₀ ✗" if r0s['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1s = "ВІДХИЛИТИ H₀ ✗" if r1s['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  [Спірмен] H₀ вірна (Y=ε)    | ρ={r0s['rho']:+.4f}  T={r0s['T']:+.4f}  "
          f"крит.=±{r0s['critical']:.3f}  → {v0s}")
    print(f"  [Спірмен] H₀ хибна (Y=X+ε) | ρ={r1s['rho']:+.4f}  T={r1s['T']:+.4f}  "
          f"крит.=±{r1s['critical']:.3f}  → {v1s}")
    
    # ── Кендалл ──
    r0k = kendall_test(n, dependent=False)
    r1k = kendall_test(n, dependent=True)
    v0k = "ВІДХИЛИТИ H₀ ✗" if r0k['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    v1k = "ВІДХИЛИТИ H₀ ✗" if r1k['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  [Кендалл] H₀ вірна (Y=ε)    | τ={r0k['tau']:+.4f}  Z={r0k['Z']:+.4f}  "
          f"крит.=±{r0k['critical']:.3f}  → {v0k}")
    print(f"  [Кендалл] H₀ хибна (Y=X+ε) | τ={r1k['tau']:+.4f}  Z={r1k['Z']:+.4f}  "
          f"крит.=±{r1k['critical']:.3f}  → {v1k}")


In [ ]:
# ── Візуалізація: хмари точок (X, Y) ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, n in enumerate(NS):
    X = np.random.uniform(0, 1, n)
    
    # Незалежні (H₀ вірна)
    Y_indep = np.random.uniform(0, 1, n)
    rho_i, _ = stats.spearmanr(X, Y_indep)
    axes[0][col].scatter(X, Y_indep, s=10, alpha=0.6, color='steelblue')
    axes[0][col].set_title(f'H₀ вірна (n={n}), ρ_s={rho_i:.3f}', fontsize=9)
    axes[0][col].set_xlabel('X'); axes[0][col].set_ylabel('Y')
    axes[0][col].grid(alpha=0.3)
    
    # Залежні (H₀ хибна)
    Y_dep = X + np.random.uniform(0, 1, n)
    rho_d, _ = stats.spearmanr(X, Y_dep)
    axes[1][col].scatter(X, Y_dep, s=10, alpha=0.6, color='salmon')
    axes[1][col].set_title(f'H₀ хибна (n={n}), ρ_s={rho_d:.3f}', fontsize=9)
    axes[1][col].set_xlabel('X'); axes[1][col].set_ylabel('Y')
    axes[1][col].grid(alpha=0.3)

plt.suptitle('Завдання 2: хмари точок (незалежні vs залежні)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Порівняння Спірмена та Кендалла ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rho_vals, tau_vals, T_sp, Z_kd = [], [], [], []
rho_f,    tau_f,    T_f,  Z_f  = [], [], [], []

for n in NS:
    r0s = spearman_test(n, False); r1s = spearman_test(n, True)
    r0k = kendall_test(n, False);  r1k = kendall_test(n, True)
    rho_vals.append(r0s['rho']); rho_f.append(r1s['rho'])
    tau_vals.append(r0k['tau']); tau_f.append(r1k['tau'])
    T_sp.append(abs(r0s['T']));  T_f.append(abs(r1s['T']))
    Z_kd.append(abs(r0k['Z']))

x = np.arange(len(NS))
w = 0.35

axes[0].bar(x - w/2, rho_vals, w, label='H₀ вірна (ρ_s)', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, rho_f,    w, label='H₀ хибна (ρ_s)', color='salmon',    alpha=0.8)
axes[0].bar(x - w/2, tau_vals, w, label='H₀ вірна (τ)', color='green', alpha=0.4, hatch='//')
axes[0].bar(x + w/2, tau_f,    w, label='H₀ хибна (τ)', color='orange', alpha=0.4, hatch='//')
axes[0].set_xticks(x); axes[0].set_xticklabels([f'n={n}' for n in NS])
axes[0].set_title('Коефіцієнти ρ_s та τ'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

axes[1].bar(x - w/2, T_sp, w, label='|T| Спірмена', color='steelblue', alpha=0.8)
axes[1].bar(x + w/2, Z_kd, w, label='|Z| Кендалла', color='green',     alpha=0.8)
# Критичні значення
for n_val, xi, crit_t, crit_z in zip(NS, x, 
        [stats.t.ppf(0.975, n-2) for n in NS],
        [stats.norm.ppf(0.975)] * 3):
    axes[1].plot([xi-w, xi], [crit_t, crit_t], 'b--', lw=1.5)
    axes[1].plot([xi, xi+w], [crit_z, crit_z], 'g--', lw=1.5)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'n={n}' for n in NS])
axes[1].set_title('Тестові статистики (H₀ вірна)'); axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

plt.suptitle('Завдання 2: Спірмен vs Кендалл', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


---
## Завдання 3. Гіпотеза випадковості: критерій на основі кількості інверсій

**Модель авторегресії AR(1):**
$$X_i = \rho \cdot X_{i-1} + \varepsilon_i, \quad \varepsilon_i \sim U[0,1]$$

При $\rho = 0$ — послідовність незалежна ($H_0$ вірна).  
При $\rho \neq 0$ — існує автокореляція ($H_0$ хибна).

**Інверсія** — пара $(i, j)$ з $i < j$ та $X_i > X_j$.

**Кількість інверсій:**
$$I = \#\{(i,j) : i < j, \, X_i > X_j\}$$

**Під $H_0$:** $\mathbb{E}[I] = \dfrac{n(n-1)}{4}$, $\;\mathbb{D}[I] = \dfrac{n(n-1)(2n+5)}{72}$

**Стандартизована статистика:** $Z = \dfrac{I - \mathbb{E}[I]}{\sqrt{\mathbb{D}[I]}} \xrightarrow{d} N(0,1)$

Перевіряємо при $\rho = $ а) 0, b) 0.5, с) 0.9.


In [ ]:
def generate_ar1(n: int, rho: float) -> np.ndarray:
    """
    Генерує послідовність довжини n за моделлю AR(1):
        X_i = ρ · X_{i-1} + ε_i,  ε_i ~ U[0,1]
    При ρ=0 — незалежні рівномірні в.в. (H₀ вірна).
    """
    eps = np.random.uniform(0, 1, n)   # рівномірний шум
    X   = np.zeros(n)
    X[0] = eps[0]                      # початкове значення
    for i in range(1, n):
        X[i] = rho * X[i - 1] + eps[i]  # рекурентна формула AR(1)
    return X


def count_inversions(X: np.ndarray) -> int:
    """
    Підраховує кількість інверсій у послідовності X:
    I = #{(i,j) : i < j, X_i > X_j}
    
    Складність O(n²) — прийнятна для n ≤ 500.
    """
    n   = len(X)
    cnt = 0
    for i in range(n):
        # Рахуємо, скільки X[j] < X[i] для j > i
        cnt += int(np.sum(X[i + 1:] < X[i]))
    return cnt


def inversions_test(n: int, rho: float) -> dict:
    """
    Критерій випадковості на основі кількості інверсій.
    
    H₀: послідовність є незалежними однаково розподіленими в.в.
    """
    # 1. Генерація послідовності
    X = generate_ar1(n, rho)
    
    # 2. Підрахунок інверсій
    I = count_inversions(X)
    
    # 3. Математичне сподівання та дисперсія під H₀
    EI = n * (n - 1) / 4
    DI = n * (n - 1) * (2 * n + 5) / 72
    
    # 4. Стандартизована статистика
    Z        = (I - EI) / np.sqrt(DI)
    critical = stats.norm.ppf(1 - ALPHA / 2)   # ≈ 1.96
    reject   = abs(Z) > critical
    
    return {"n": n, "rho": rho, "I": I, "EI": EI, "DI": DI,
            "Z": Z, "critical": critical, "reject": reject}

# ── Друк результатів ────────────────────────────────────────────────────────
print("=" * 90)
print("ЗАВДАННЯ 3. КРИТЕРІЙ ВИПАДКОВОСТІ (ІНВЕРСІЇ)")
print("=" * 90)

RHOS = [0.0, 0.5, 0.9]
N_INV = 100   # розмір вибірки для тесту на інверсіях

for rho in RHOS:
    r = inversions_test(N_INV, rho)
    v = "ВІДХИЛИТИ H₀ ✗" if r['reject'] else "НЕ ВІДХИЛИТИ H₀ ✓"
    print(f"  ρ = {rho:.1f} | I={r['I']:5d}  E[I]={r['EI']:.0f}  "
          f"Z={r['Z']:+.4f}  крит.=±{r['critical']:.3f}  → {v}")


In [ ]:
# ── Детальніший аналіз: різні n та ρ ────────────────────────────────────────
print("=" * 90)
print("ВПЛИВ n ТА ρ НА СТАТИСТИКУ ІНВЕРСІЙ")
print("=" * 90)
print(f"  {'n':>5}  {'ρ':>5}  {'I':>8}  {'E[I]':>8}  {'Z':>8}  {'Рішення'}")
print(f"  {'─'*5}  {'─'*5}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*20}")

for n in NS:
    for rho in RHOS:
        r = inversions_test(n, rho)
        v = "ВІДХИЛЕНО" if r['reject'] else "не відхилено"
        print(f"  {n:>5}  {rho:>5.1f}  {r['I']:>8d}  {r['EI']:>8.0f}  "
              f"{r['Z']:>+8.4f}  {v}")
    print()


In [ ]:
# ── Візуалізація: послідовності та розподіл статистики Z ────────────────────
fig = plt.figure(figsize=(16, 9))

# Верхній ряд: послідовності AR(1) для різних ρ
n_plot = 100
for col, rho in enumerate(RHOS):
    ax = fig.add_subplot(2, 3, col + 1)
    X  = generate_ar1(n_plot, rho)
    ax.plot(X, color=['steelblue', 'orange', 'red'][col], lw=1, alpha=0.8)
    ax.set_title(f'AR(1), ρ={rho}', fontsize=10)
    ax.set_xlabel('i'); ax.set_ylabel('$X_i$'); ax.grid(alpha=0.3)

# Нижній ряд: гістограми Z по 200 симуляціях
n_sim = 200
for col, rho in enumerate(RHOS):
    ax    = fig.add_subplot(2, 3, col + 4)
    Z_sim = [inversions_test(n_plot, rho)['Z'] for _ in range(n_sim)]
    
    ax.hist(Z_sim, bins=25, density=True, alpha=0.7,
            color=['steelblue', 'orange', 'red'][col], edgecolor='black', lw=0.4)
    
    # Накладаємо теоретичний N(0,1)
    x_g = np.linspace(min(Z_sim) - 1, max(Z_sim) + 1, 300)
    ax.plot(x_g, stats.norm.pdf(x_g), 'k-', lw=2, label='N(0,1)')
    
    # Критичні межі
    crit = stats.norm.ppf(1 - ALPHA / 2)
    ax.axvline( crit, color='green', ls='--', lw=1.5, label=f'крит.=±{crit:.2f}')
    ax.axvline(-crit, color='green', ls='--', lw=1.5)
    
    reject_rate = np.mean(np.abs(Z_sim) > crit)
    ax.set_title(f'Z при ρ={rho} (відх. {reject_rate:.0%})', fontsize=9)
    ax.set_xlabel('Z'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle('Завдання 3: послідовності AR(1) та розподіл статистики Z', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Висновки

### Завдання 1 (Пусті блоки — однорідність)
- При $H_0$ вірній (обидві вибірки з $U[0,1]$): статистика $Z$ знаходиться в критичній зоні рідко.  
- При $H_0$ хибній ($Y \sim U[0,2]$): кількість пустих блоків різко зростає, $|Z| \gg z_{\alpha/2}$.  
- Зі збільшенням $n$ тест стає точнішим.

### Завдання 2 (Незалежність — Спірмен та Кендалл)
- Обидва критерії узгоджено **не відхиляють** $H_0$ для незалежних $X, Y$.  
- Обидва надійно **відхиляють** $H_0$ при $Y = X + \varepsilon$ (корельовані).  
- Коефіцієнт Спірмена $\rho_s \approx 0.7$, Кендалла $\tau \approx 0.5$ — обидва значущі.  
- Зі збільшенням $n$ тести стають потужнішими.

### Завдання 3 (Випадковість — інверсії)
- При $\rho = 0$: $Z \approx 0$ — $H_0$ не відхиляється.  
- При $\rho = 0.5$: позитивна автокореляція → менше інверсій → $Z > 0$, $H_0$ відхиляється.  
- При $\rho = 0.9$: сильна автокореляція → $|Z| \gg 1.96$, $H_0$ надійно відхиляється.
